In [1]:
import os
import json
import urllib.parse
import requests
from typing import List, Optional, Dict
from pydantic import BaseModel
from google import genai
from google.genai import types

In [2]:
class ApiQueryParameters(BaseModel):
    name: str
    population: Optional[str] = None
    severity: Optional[str] = None
    type: Optional[str] = None

In [3]:
class ApiQueries(BaseModel):
    queries: List[ApiQueryParameters]

In [4]:
def get_extraction_instructions() -> str:
    return """
    You are an expert clinical data extraction assistant.
    Extract the required biomarkers from the clinical case to build query parameters based on the system rules.

    Rules:
    1. name: Extract all laboratory indicators and biomarkers mentioned.
    2. population: Infer from patient age. Allowed values: adult, children, neonatal.
    3. severity: Infer from clinical description. Allowed values: SIRS, sepsis, severe, shock, syndrome. Leave null if unsure.
    4. type: Infer if it is for diagnosis or prognosis. Initial cases are usually diagnosis. Leave null if unsure.
    """

In [5]:
def extract_parameters(client: genai.Client, clinical_case: str) -> ApiQueries:
    prompt = f"Analyze the following clinical case and extract the search parameters:\n\n{clinical_case}"
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[get_extraction_instructions(), prompt],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ApiQueries,
            temperature=0.0
        )
    )
    
    try:
        parsed_data = json.loads(response.text)
        return ApiQueries(**parsed_data)
    except Exception as e:
        print(f"Error parsing model extraction response: {e}")
        return ApiQueries(queries=[])

In [6]:
def build_urls(api_queries: ApiQueries) -> Dict[str, str]:
    base_url = "http://sysbio.org.cn/SBD/api.php"
    result_urls = {}
    
    for query in api_queries.queries:
        params = {}
        if query.name:
            params['name'] = query.name
        if query.population:
            params['population'] = query.population
        if query.severity:
            params['severity'] = query.severity
        if query.type:
            params['type'] = query.type
            
        query_string = urllib.parse.urlencode(params, quote_via=urllib.parse.quote)
        result_urls[query.name] = f"{base_url}?{query_string}"
        
    return result_urls

In [7]:
def fetch_database_records(api_urls: Dict[str, str]) -> str:
    reference_data = ""
    for biomarker, url in api_urls.items():
        try:
            print(f"-> Fetching database records for {biomarker}...")
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                data = response.json()
                results = data.get("results", []) if isinstance(data, dict) else data
                if not results:
                    results = data 
                
                reference_data += f"\n--- {biomarker} Database Records ---\n"
                reference_data += json.dumps(results[:2], indent=2, ensure_ascii=False)
            else:
                print(f"-> Failed to fetch {biomarker}. HTTP Status: {response.status_code}")
        except Exception as e:
            print(f"-> Exception occurred while fetching {biomarker}: {e}")

    return reference_data

In [8]:
def generate_final_diagnosis(client: genai.Client, clinical_case: str, reference_data: str) -> str:
    prompt = f"""You are an expert clinical AI assistant.

    Here is some reference data retrieved automatically from a professional biomarker database regarding diagnostic standards:
    <reference_data>
    {reference_data}
    </reference_data>

    Based ON THE REFERENCE DATA ABOVE and your medical knowledge, please analyze the following clinical case:

    <clinical_case>
    {clinical_case}
    </clinical_case>

    Task:
    1. Compare the patient biomarker values with the cut-off values in the reference data.
    2. Can this patient be diagnosed with sepsis?
    3. Please provide the percentage of possibility that this patient is diagnosed with sepsis.
    4. What is the best treatment plan?
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.2
        )
    )
    return response.text

In [9]:
def run_diagnostic_pipeline():
    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        print("Fatal Error: Please set the GOOGLE_API_KEY environment variable before running.")
        return

    client = genai.Client(api_key=api_key)
    
    print("==================================================")
    print("End-to-End Automated Sepsis Diagnostic Pipeline")
    print("Type QUIT to exit the application.")
    print("==================================================")
    
    while True:
        print("\nPlease enter the clinical case details:")
        
        lines = []
        while True:
            try:
                line = input()
            except EOFError:
                break
                
            if line.strip().upper() == "QUIT":
                print("Exiting application safely.")
                return
                
            if line.strip() == "</clinical_case>":
                lines.append(line)
                break
                
            if not line.strip() and len(lines) > 0 and not lines[0].startswith("<clinical_case>"):
                break
                
            lines.append(line)
            
        case_text = "\n".join(lines)
        if not case_text.strip():
            continue
            
        print("\n[Phase 1] Analyzing case and generating database queries...")
        queries = extract_parameters(client, case_text)
        urls = build_urls(queries)
        
        if not urls:
            print("Warning: No relevant biomarkers found to query.")
            continue
            
        print("\n--- Generated Database Queries ---")
        for name, link in urls.items():
            print(f"[{name}]: {link}")
            
        print("\n[Phase 2] Retrieving evidence from external knowledge hub...")
        fetched_evidence = fetch_database_records(urls)
        
        if not fetched_evidence.strip():
            print("Warning: Retrieved data is empty. Relying solely on internal model weights.")

        print("\n[Phase 3] Generating final clinical assessment...")
        final_report = generate_final_diagnosis(client, case_text, fetched_evidence)
        
        print("\n================ DIAGNOSTIC REPORT ================\n")
        print(final_report)
        print("\n===================================================\n")

if __name__ == "__main__":
    run_diagnostic_pipeline()

Fatal Error: Please set the GOOGLE_API_KEY environment variable before running.
